# Hybrid FITS-normalized DC4 point-source catalog generator

This notebook creates a fully fixed astromodels YAML catalog for the NGC 4151
analysis. Spectral shapes come from the injected analytic or `.dat` models,
while each source amplitude is calibrated against its native simulated FITS
events.

Steady-source normalizations are fitted using the full three-month events and
full three-month orientation. Their values are independent of the NGC 4151
pointing GTI. Variable and flaring sources are instead fitted using only events
inside the NGC 4151 60° GTI and the ordinary, unweighted GTI orientation. Their
frozen amplitudes therefore represent their time-averaged contributions inside
that particular GTI. The fitting notebook must use the same GTI and must not
apply those light curves again.


In [ ]:
from copy import deepcopy
from pathlib import Path

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from astromodels import (
    Cutoff_powerlaw,
    Line,
    LinearPolarization,
    Log_parabola,
    Model,
    PointSource,
    Powerlaw,
    SpectralComponent,
    load_model,
)
from histpy import Axis, Axes, HealpixAxis, Histogram

from cosipy.data_io import BinnedData, EmCDSBinnedData
from cosipy.event_selection import GoodTimeInterval
from cosipy.response import (
    BinnedInstrumentResponse,
    BinnedThreeMLModelFolding,
    BinnedThreeMLPointSourceResponse,
)
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.threeml.custom_functions import SpecFromDat

%matplotlib inline


## Locate the injected spectral tables

The tables are read directly from the local `cositools/cosi-sim` clone at `/Users/parshadkp/Software/cosi-sim/cosi_sim/Source_Library`. The helper searches recursively below its `DC3/` and `DC4/` directories.

The catalog uses the following injected tables where a regular astromodels function is not a faithful representation:

- `cygX1_hard_0.1-0.4_spec.dat`
- `cygX1_hard_0.4-10_spec.dat`
- `GRS1758_spec.dat`
- `Crab_Ne_spec.dat`, `Crab_P1_spec.dat`, `Crab_Brg_spec.dat`, and `Crab_P2_spec.dat`
- `4C71p07_spectrum.dat`
- `4C_21_spectrum_noflare.dat` and `4C_21_spectrum_flare.dat`
- `MAXIJ1820_0.1-0.4_spec.dat` and `MAXIJ1820_0.4-10_spec.dat`
- `MAXIJ1348_0.1-0.4_spec.dat` and `MAXIJ1348_0.4-10_spec.dat`
- `3C454p3_low_spectrum.dat` and `3C454p3_high_spectrum.dat`
- `cygX3_transition_spec.dat`, `PSRB1259_spec.dat`, `LS5039_spec.dat`, and `J1846_spec.dat`
- `nova_co_continuum_spec.dat`

The individual simulated FITS source files in `DC4_Files/DC4_Sources` are event-data products and cannot replace these spectral `.dat` files.

In [ ]:
SOURCE_DATA_ROOT = Path(
    "/Users/parshadkp/Software/cosi-sim/cosi_sim/Source_Library"
)


def find_source_file(challenge, filename):
    """Find one source-library file below DC3/ or DC4/."""
    search_root = SOURCE_DATA_ROOT / challenge
    matches = sorted(search_root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename!r} below {search_root}. "
            "The simulated FITS source files are not spectral tables; copy the "
            "corresponding .dat file from the cosi-sim source library into this "
            "DC3/DC4 directory."
        )

    if len(matches) > 1:
        raise RuntimeError(
            f"Found more than one copy of {filename!r}: {matches}. "
            "Keep one copy or set the desired path explicitly."
        )

    return matches[0].resolve()


cygx1_low_path = find_source_file("DC3", "cygX1_hard_0.1-0.4_spec.dat")
cygx1_high_path = find_source_file("DC3", "cygX1_hard_0.4-10_spec.dat")
grs1758_path = find_source_file("DC3", "GRS1758_spec.dat")
crab_nebula_path = find_source_file("DC4", "Crab_Ne_spec.dat")
crab_p1_path = find_source_file("DC4", "Crab_P1_spec.dat")
crab_bridge_path = find_source_file("DC4", "Crab_Brg_spec.dat")
crab_p2_path = find_source_file("DC4", "Crab_P2_spec.dat")
four_c_71p07_path = find_source_file("DC4", "4C71p07_spectrum.dat")
four_c_21p35_noflare_path = find_source_file(
    "DC3", "4C_21_spectrum_noflare.dat"
)
four_c_21p35_flare_path = find_source_file(
    "DC3", "4C_21_spectrum_flare.dat"
)
maxi_j1820_low_path = find_source_file("DC3", "MAXIJ1820_0.1-0.4_spec.dat")
maxi_j1820_high_path = find_source_file("DC3", "MAXIJ1820_0.4-10_spec.dat")
maxi_j1348_low_path = find_source_file("DC3", "MAXIJ1348_0.1-0.4_spec.dat")
maxi_j1348_high_path = find_source_file("DC3", "MAXIJ1348_0.4-10_spec.dat")
three_c_454p3_low_path = find_source_file("DC4", "3C454p3_low_spectrum.dat")
three_c_454p3_high_path = find_source_file("DC4", "3C454p3_high_spectrum.dat")
cyg_x3_path = find_source_file("DC4", "cygX3_transition_spec.dat")
psr_b1259_path = find_source_file("DC3", "PSRB1259_spec.dat")
ls_5039_path = find_source_file("DC3", "LS5039_spec.dat")
psr_j1846_path = find_source_file("DC3", "J1846_spec.dat")
co_nova_continuum_path = find_source_file("DC3", "nova_co_continuum_spec.dat")

print("Cyg X-1 low-energy table:", cygx1_low_path)
print("Cyg X-1 high-energy table:", cygx1_high_path)
print("GRS 1758 table:", grs1758_path)
print("Crab tables:", crab_nebula_path, crab_p1_path, crab_bridge_path, crab_p2_path)
print("4C 71.07 table:", four_c_71p07_path)
print("4C 21.35 tables:", four_c_21p35_noflare_path, four_c_21p35_flare_path)
print("MAXI J1820 tables:", maxi_j1820_low_path, maxi_j1820_high_path)
print("MAXI J1348 tables:", maxi_j1348_low_path, maxi_j1348_high_path)
print("3C 454.3 tables:", three_c_454p3_low_path, three_c_454p3_high_path)
print("Cyg X-3 table:", cyg_x3_path)
print("PSR B1259 table:", psr_b1259_path)
print("LS 5039 table:", ls_5039_path)
print("PSR J1846 table:", psr_j1846_path)
print("CO nova continuum table:", co_nova_continuum_path)

## Table-spectrum helper

`SpecFromDat` preserves the tabulated shape and provides a normalization parameter. Its current normalization uses a simple bin-width sum, so the helper below performs a numerical correction such that the model integral from 100 keV to 10 MeV equals the injected source-file flux. Absolute paths are stored in the YAML so the model can be reloaded from another working directory. The normalization is fixed before serialization.

In [ ]:
ENERGY_MIN = 100.0
ENERGY_MAX = 10_000.0
PHOTON_FLUX_UNIT = 1 / (u.cm**2 * u.s)
DIFFERENTIAL_FLUX_UNIT = 1 / (u.keV * u.cm**2 * u.s)


def table_spectrum(dat_path, injected_photon_flux):
    """Create a SpecFromDat component normalized to an integrated flux."""
    spectrum = SpecFromDat(
        K=float(injected_photon_flux),
        dat=Path(dat_path).resolve(),
    )

    energy = np.geomspace(ENERGY_MIN, ENERGY_MAX, 20_000)
    current_integral = np.trapezoid(spectrum(energy), energy)

    if not np.isfinite(current_integral) or current_integral <= 0:
        raise ValueError(f"Invalid spectrum integral for {dat_path}: {current_integral}")

    spectrum.K.value *= float(injected_photon_flux) / current_integral
    return spectrum


def normalize_analytic_components(components, injected_photon_flux):
    """Scale analytic component amplitudes to the injected integrated flux."""
    energy = np.geomspace(ENERGY_MIN, ENERGY_MAX, 20_000)
    current_integral = np.trapezoid(
        sum(component(energy) for component in components),
        energy,
    )

    if not np.isfinite(current_integral) or current_integral <= 0:
        raise ValueError(f"Invalid analytic spectrum integral: {current_integral}")

    scale = float(injected_photon_flux) / current_integral
    for component in components:
        component.K.value *= scale


def set_normalization_step(parameter):
    """Use a scale-appropriate initial optimizer step."""
    parameter.delta = max(abs(parameter.value) * 0.05, 1e-12)

## 1. Cyg X-1 hard state

The injected hard-state spectrum is an `eqpair` calculation divided into two energy ranges for the polarization simulation. Native astromodels does not provide this `eqpair` model, so the two injected tables are retained and their relative normalization is fixed.

In [ ]:
cygx1_low_flux = 0.04243172227636306
cygx1_high_flux = 0.003371822180706115

cygx1_low = table_spectrum(cygx1_low_path, cygx1_low_flux)
cygx1_high = table_spectrum(cygx1_high_path, cygx1_high_flux)
cygx1_spectrum = cygx1_low + cygx1_high

cygx1 = PointSource(
    "cyg_x1_hard",
    l=71.33496,
    b=3.066917,
    spectral_shape=cygx1_spectrum,
)

## 2. Crab

The DC4 Crab is retained as four distinct astromodels spectral components:
nebula, peak 1, bridge, and peak 2. Each uses its exact injected table and
integrated flux. The nebula has 40% polarization at 160° and the three pulsar
components have 20% polarization at 145°, in Galactic/IAU coordinates.

The pulsar light curves repeat every 0.0333924123 s and are normalized to unit
mean by MEGAlib. Each 15 s spacecraft interval therefore averages over about
449 pulse periods, so the time-integrated response uses their mean spectra.
The catalog preserves the injected polarization metadata, but exact
polarized folding additionally requires a detector response with a `Pol` axis.
The current continuum response lacks that axis, so the fitting notebook reports
and uses an in-memory zero-polarization approximation.

Although `Crab.source` contains `EarthOccultation false`, transferring that flag
to the current binned response overpredicts the selected Crab counts by almost
an order of magnitude. The spectral fit therefore retains the response's
standard source-visibility treatment, which agrees with the selected DC4 Crab
events.


In [ ]:
crab_nebula = table_spectrum(crab_nebula_path, 0.033197515)
crab_p1 = table_spectrum(crab_p1_path, 0.002380673782066656)
crab_bridge = table_spectrum(crab_bridge_path, 0.0006280494007254569)
crab_p2 = table_spectrum(crab_p2_path, 0.0032012384701172636)

crab_components = [
    SpectralComponent(
        "nebula",
        crab_nebula,
        LinearPolarization(40.0, 160.0),
    ),
    SpectralComponent(
        "peak1",
        crab_p1,
        LinearPolarization(20.0, 145.0),
    ),
    SpectralComponent(
        "bridge",
        crab_bridge,
        LinearPolarization(20.0, 145.0),
    ),
    SpectralComponent(
        "peak2",
        crab_p2,
        LinearPolarization(20.0, 145.0),
    ),
]

crab = PointSource(
    "crab",
    l=184.5575,
    b=-5.78434,
    components=crab_components,
)

for component in crab.components.values():
    component.polarization.degree.fix = True
    component.polarization.angle.fix = True


## 3. 1E 1740.7−2942

The injected `compow` table is accurately represented in the COSI energy range by a cutoff power law plus a high-energy power-law tail. Their relative normalization is linked, and the remaining overall amplitude is fixed before the catalog is saved.

In [ ]:
one_e_thermal = Cutoff_powerlaw()
one_e_thermal.K.value = 7.871196815455384e-4
one_e_thermal.piv.value = 300.0
one_e_thermal.index.value = 0.4379077571700285
one_e_thermal.xc.value = 31.912756594453995

one_e_tail = Powerlaw()
one_e_tail.K.value = 3.929675541578979e-6
one_e_tail.piv.value = 300.0
one_e_tail.index.value = -1.8998638815533166

for spectrum in (one_e_thermal, one_e_tail):
    spectrum.K.unit = DIFFERENTIAL_FLUX_UNIT
    spectrum.piv.unit = u.keV

one_e_thermal.xc.unit = u.keV

for parameter in (
    one_e_thermal.piv,
    one_e_thermal.index,
    one_e_thermal.xc,
    one_e_tail.piv,
    one_e_tail.index,
):
    parameter.fix = True

one_e_spectrum = one_e_thermal + one_e_tail

one_e_1740 = PointSource(
    "one_e_1740_compow",
    l=359.11596,
    b=-0.10575,
    spectral_shape=one_e_spectrum,
)

## 4. GRS 1758−258

The injected spectrum is a thermal-Comptonization model. The original table is retained rather than replacing its curvature with a simple power law.

In [ ]:
grs1758_flux = 0.003495
grs1758_spectrum = table_spectrum(grs1758_path, grs1758_flux)

grs1758 = PointSource(
    "grs_1758_258",
    l=4.50780,
    b=-1.36106,
    spectral_shape=grs1758_spectrum,
)

## 5. Cen A

Cen A is injected as a power law with photon index −1.732 and integrated 100 keV–10 MeV flux 0.00197 ph cm$^{-2}$ s$^{-1}$.

In [ ]:
cena_spectrum = Powerlaw()
cena_spectrum.K.value = 2.227338148135328e-6
cena_spectrum.piv.value = 300.0
cena_spectrum.index.value = -1.732

cena_spectrum.K.unit = DIFFERENTIAL_FLUX_UNIT
cena_spectrum.piv.unit = u.keV
cena_spectrum.piv.fix = True
cena_spectrum.index.fix = True
cena = PointSource(
    "cena",
    l=309.516,
    b=19.417,
    spectral_shape=cena_spectrum,
)

## 6. 4C 71.07

4C 71.07 retains its injected tabulated spectrum and is treated as steady.

In [ ]:
four_c_71p07_spectrum = table_spectrum(four_c_71p07_path, 0.00119)
four_c_71p07 = PointSource(
    "four_c_71p07",
    l=143.540759,
    b=34.425671,
    spectral_shape=four_c_71p07_spectrum,
)

## 7. 4C 21.35

The non-flare and flare spectra are separate variable catalog entries. Each is fitted with the ordinary NGC 4151 GTI response, so its frozen amplitude is the component's time-averaged contribution inside this GTI.

In [ ]:
four_c_21p35_noflare_spectrum = table_spectrum(
    four_c_21p35_noflare_path,
    0.0005535,
)
four_c_21p35_noflare = PointSource(
    "four_c_21p35_noflare",
    l=255.073637,
    b=81.659766,
    spectral_shape=four_c_21p35_noflare_spectrum,
)

four_c_21p35_flare_spectrum = table_spectrum(
    four_c_21p35_flare_path,
    0.017293201590129727,
)
four_c_21p35_flare = PointSource(
    "four_c_21p35_flare",
    l=255.073637,
    b=81.659766,
    spectral_shape=four_c_21p35_flare_spectrum,
)

## 8. MAXI J1820

The two injected tabulated components share one outburst light curve and one linked catalog amplitude.

In [ ]:
maxi_j1820_low = table_spectrum(
    maxi_j1820_low_path,
    0.13820979415525897,
)
maxi_j1820_high = table_spectrum(
    maxi_j1820_high_path,
    0.005963517352694539,
)
maxi_j1820 = PointSource(
    "maxi_j1820",
    l=35.8535,
    b=10.15915,
    spectral_shape=maxi_j1820_low + maxi_j1820_high,
)

## 9. MAXI J1348−630

The two injected tabulated components share one outburst light curve and one linked catalog amplitude.

In [ ]:
maxi_j1348_low = table_spectrum(
    maxi_j1348_low_path,
    0.08633295828868433,
)
maxi_j1348_high = table_spectrum(
    maxi_j1348_high_path,
    0.0023113717669982913,
)
maxi_j1348 = PointSource(
    "maxi_j1348",
    l=309.26389732,
    b=-1.10328,
    spectral_shape=maxi_j1348_low + maxi_j1348_high,
)

## 10. 3C 454.3

The low and high states are separate variable catalog entries. Their normalizations are fitted independently from their native FITS events inside the NGC 4151 GTI.

In [ ]:
three_c_454p3_low = PointSource(
    "three_c_454p3_low",
    l=86.111069,
    b=-38.183817,
    spectral_shape=table_spectrum(three_c_454p3_low_path, 0.00029),
)
three_c_454p3_high = PointSource(
    "three_c_454p3_high",
    l=86.111069,
    b=-38.183817,
    spectral_shape=table_spectrum(three_c_454p3_high_path, 0.00624),
)

## 11. NGC 1068

NGC 1068 uses its injected cutoff-power-law plus high-energy power-law tail. The components are normalized to the injected 100 keV–10 MeV photon flux and linked at their injected ratio.

In [ ]:
ngc1068_thermal = Cutoff_powerlaw()
ngc1068_thermal.K.value = 0.308 * 200.0**-1.92
ngc1068_thermal.piv.value = 200.0
ngc1068_thermal.index.value = -1.92
ngc1068_thermal.xc.value = 200.0
ngc1068_tail = Powerlaw()
ngc1068_tail.K.value = 91.18 * 200.0**-3.8
ngc1068_tail.piv.value = 200.0
ngc1068_tail.index.value = -3.8

for component in (ngc1068_thermal, ngc1068_tail):
    component.K.unit = DIFFERENTIAL_FLUX_UNIT
    component.piv.unit = u.keV
    component.piv.fix = True
    component.index.fix = True

ngc1068_thermal.xc.unit = u.keV
ngc1068_thermal.xc.fix = True
normalize_analytic_components((ngc1068_thermal, ngc1068_tail), 0.00161)

ngc1068 = PointSource(
    "ngc1068",
    l=172.103584,
    b=-51.933771,
    spectral_shape=ngc1068_thermal + ngc1068_tail,
)

## 12. NGC 4151

This is the updated mock-data injection from `AGN_Corona_DC4.ipynb`: a cutoff power law with `K = 0.15` at 1 keV, index −1.75, and cutoff 200 keV, plus an index −3.8 power-law tail whose differential flux at 200 keV is 30% of the thermal flux. The implementation is pivoted at 200 keV without changing the injected spectrum.

In [ ]:
ngc4151_thermal = Cutoff_powerlaw()
ngc4151_thermal.K.value = 0.15 * 200.0**-1.75
ngc4151_thermal.piv.value = 200.0
ngc4151_thermal.index.value = -1.75
ngc4151_thermal.xc.value = 200.0
ngc4151_tail = Powerlaw()
ngc4151_tail_fraction_at_200kev = 0.30
ngc4151_tail.K.value = (
    ngc4151_tail_fraction_at_200kev
    * ngc4151_thermal.evaluate_at(200.0)
)
ngc4151_tail.piv.value = 200.0
ngc4151_tail.index.value = -3.8

for component in (ngc4151_thermal, ngc4151_tail):
    component.K.unit = DIFFERENTIAL_FLUX_UNIT
    component.piv.unit = u.keV
    component.piv.fix = True
    component.index.fix = True

ngc4151_thermal.xc.unit = u.keV
ngc4151_thermal.xc.fix = True
assert np.isclose(
    ngc4151_tail.evaluate_at(200.0)
    / ngc4151_thermal.evaluate_at(200.0),
    ngc4151_tail_fraction_at_200kev,
)

ngc4151_energy = np.geomspace(ENERGY_MIN, ENERGY_MAX, 20_000)
ngc4151_injected_flux = np.trapezoid(
    (ngc4151_thermal + ngc4151_tail)(ngc4151_energy),
    ngc4151_energy,
)
assert np.isclose(ngc4151_injected_flux, 0.0025171894, rtol=1e-5)

ngc4151 = PointSource(
    "ngc4151",
    l=155.077404,
    b=75.063170,
    spectral_shape=ngc4151_thermal + ngc4151_tail,
)

## 13. Cyg X-3

The injected transition-state spectrum is retained as a tabulated model.

In [ ]:
cyg_x3 = PointSource(
    "cyg_x3",
    l=79.84549,
    b=0.70006,
    spectral_shape=table_spectrum(cyg_x3_path, 0.001331405),
)

## 14. PSR B1259−63

The injected table defines the spectral shape. Its normalization is fitted from native FITS events inside the NGC 4151 GTI, absorbing the light-curve average for this selection.

In [ ]:
psr_b1259 = PointSource(
    "psr_b1259",
    l=304.18358257,
    b=-0.99158457,
    spectral_shape=table_spectrum(psr_b1259_path, 0.00061253),
)

## 15. 1RXS J170849.0−400901

The injected magnetar continuum is represented directly with a fixed astromodels `Log_parabola`.

In [ ]:
one_rxs_j170849_spectrum = Log_parabola()
one_rxs_j170849_spectrum.K.value = 1.68e-6
one_rxs_j170849_spectrum.piv.value = 143.276
one_rxs_j170849_spectrum.alpha.value = -1.637
one_rxs_j170849_spectrum.beta.value = 0.261
one_rxs_j170849_spectrum.K.unit = DIFFERENTIAL_FLUX_UNIT
one_rxs_j170849_spectrum.piv.unit = u.keV
one_rxs_j170849_spectrum.piv.fix = True
one_rxs_j170849_spectrum.alpha.fix = True
one_rxs_j170849_spectrum.beta.fix = True
normalize_analytic_components((one_rxs_j170849_spectrum,), 0.00033)
one_rxs_j170849 = PointSource(
    "one_rxs_j170849",
    l=346.47938142,
    b=0.03838608,
    spectral_shape=one_rxs_j170849_spectrum,
)

## 16. Generic magnetar

The generic magnetar uses the injected fixed `Log_parabola`, with twice the curvature parameter of the 1RXS J170849.0−400901 model.

In [ ]:
generic_magnetar_spectrum = Log_parabola()
generic_magnetar_spectrum.K.value = 1.68e-6
generic_magnetar_spectrum.piv.value = 143.276
generic_magnetar_spectrum.alpha.value = -1.637
generic_magnetar_spectrum.beta.value = 0.522
generic_magnetar_spectrum.K.unit = DIFFERENTIAL_FLUX_UNIT
generic_magnetar_spectrum.piv.unit = u.keV
generic_magnetar_spectrum.piv.fix = True
generic_magnetar_spectrum.alpha.fix = True
generic_magnetar_spectrum.beta.fix = True
normalize_analytic_components((generic_magnetar_spectrum,), 0.00033)

generic_magnetar = PointSource(
    "generic_magnetar",
    l=250.0,
    b=0.03838608,
    spectral_shape=generic_magnetar_spectrum,
)

## 17. LS 5039

The injected table defines the spectral shape. Its normalization is fitted from native FITS events inside the NGC 4151 GTI, absorbing the periodic average for this selection.

In [ ]:
ls_5039 = PointSource(
    "ls_5039",
    l=16.88158741,
    b=-1.28921165,
    spectral_shape=table_spectrum(ls_5039_path, 0.0003006),
)

## 18. PSR J1846−0258

The injected table defines the spectral shape. Its normalization is fitted from native FITS events inside the NGC 4151 GTI, absorbing the phase average for this selection.

In [ ]:
psr_j1846 = PointSource(
    "psr_j1846",
    l=29.71195,
    b=-0.24012,
    spectral_shape=table_spectrum(psr_j1846_path, 0.000152392),
)

## 19. CO nova continuum

Only the continuum component is included here. Its normalization is fitted from native FITS events inside the NGC 4151 GTI, so the frozen amplitude absorbs the short nova duration in this selection.

In [ ]:
co_nova_continuum = PointSource(
    "co_nova_continuum",
    l=310.9847,
    b=2.7256,
    spectral_shape=table_spectrum(co_nova_continuum_path, 0.122071),
)

## Choose sources by name

NGC 4151 is always the fit target. Set `ADDITIONAL_SOURCE_NAMES = []` for an
NGC-4151-only catalog, or enter one or more exact names, for example:

```python
ADDITIONAL_SOURCE_NAMES = ["crab"]
ADDITIONAL_SOURCE_NAMES = ["crab", "cena", "ngc1068"]
```

The output filename includes the total number of sources in the model,
including NGC 4151. For example, any 20-source selection is saved as
`source_catalog_DC4_20sources.yaml`. The two states of 4C 21.35 and 3C 454.3
are separate entries because each state has a different injected spectrum and
light curve.


In [ ]:
# Enter the exact nuisance-source names to include. NGC 4151 is automatic.
# Examples: ["crab"] or ["crab", "cena", "ngc1068"].
ADDITIONAL_SOURCE_NAMES = [
    "crab",
    "four_c_21p35_flare",
    "maxi_j1820",
    "maxi_j1348",
    "four_c_71p07",
    "four_c_21p35_noflare",
    "cena",
    "cyg_x1_hard",
    "one_e_1740_compow",
    "generic_magnetar",
    "psr_b1259",
    "grs_1758_258",
    "ls_5039",
    "one_rxs_j170849",
    "ngc1068",
    "cyg_x3",
    "psr_j1846",
    "three_c_454p3_low",
    "three_c_454p3_high",
    "co_nova_continuum",]

# Valid names, ordered by approximate contribution in the NGC 4151 FOV.
AVAILABLE_SOURCE_NAMES = [
    "crab",
    "four_c_21p35_flare",
    "maxi_j1820",
    "maxi_j1348",
    "four_c_71p07",
    "four_c_21p35_noflare",
    "cena",
    "cyg_x1_hard",
    "one_e_1740_compow",
    "generic_magnetar",
    "psr_b1259",
    "grs_1758_258",
    "ls_5039",
    "one_rxs_j170849",
    "ngc1068",
    "cyg_x3",
    "psr_j1846",
    "three_c_454p3_low",
    "three_c_454p3_high",
    "co_nova_continuum",
]

all_catalog_sources = {
    "cyg_x1_hard": cygx1,
    "crab": crab,
    "one_e_1740_compow": one_e_1740,
    "grs_1758_258": grs1758,
    "cena": cena,
    "four_c_71p07": four_c_71p07,
    "four_c_21p35_noflare": four_c_21p35_noflare,
    "four_c_21p35_flare": four_c_21p35_flare,
    "maxi_j1820": maxi_j1820,
    "maxi_j1348": maxi_j1348,
    "three_c_454p3_low": three_c_454p3_low,
    "three_c_454p3_high": three_c_454p3_high,
    "ngc1068": ngc1068,
    "ngc4151": ngc4151,
    "cyg_x3": cyg_x3,
    "psr_b1259": psr_b1259,
    "one_rxs_j170849": one_rxs_j170849,
    "generic_magnetar": generic_magnetar,
    "ls_5039": ls_5039,
    "psr_j1846": psr_j1846,
    "co_nova_continuum": co_nova_continuum,
}

if set(AVAILABLE_SOURCE_NAMES) != set(all_catalog_sources) - {"ngc4151"}:
    raise RuntimeError(
        "AVAILABLE_SOURCE_NAMES does not contain every nuisance entry"
    )

if isinstance(ADDITIONAL_SOURCE_NAMES, str):
    requested_source_names = [ADDITIONAL_SOURCE_NAMES]
else:
    requested_source_names = list(ADDITIONAL_SOURCE_NAMES)

if "ngc4151" in requested_source_names:
    raise ValueError(
        "Do not include 'ngc4151' in ADDITIONAL_SOURCE_NAMES; "
        "the target is included automatically."
    )

duplicate_source_names = sorted(
    {
        name
        for name in requested_source_names
        if requested_source_names.count(name) > 1
    }
)
if duplicate_source_names:
    raise ValueError(
        "Duplicate names in ADDITIONAL_SOURCE_NAMES: "
        + ", ".join(duplicate_source_names)
    )

unknown_source_names = sorted(
    set(requested_source_names) - set(AVAILABLE_SOURCE_NAMES)
)
if unknown_source_names:
    raise ValueError(
        "Unknown source name(s): "
        + ", ".join(unknown_source_names)
        + ". Valid names are: "
        + ", ".join(AVAILABLE_SOURCE_NAMES)
    )

selected_source_names = ["ngc4151", *requested_source_names]
model = Model(*(all_catalog_sources[name] for name in selected_source_names))


def link_at_current_ratio(dependent, independent):
    relation = Line(a=0.0, b=dependent.value / independent.value)
    relation.a.fix = True
    relation.b.fix = True
    model.link(dependent, independent, relation)


if "cyg_x1_hard" in model.sources:
    link_at_current_ratio(
        model.cyg_x1_hard.spectrum.main.composite.K_2,
        model.cyg_x1_hard.spectrum.main.composite.K_1,
    )

if "crab" in model.sources:
    crab_nebula_k = model["crab.spectrum.nebula.SpecFromDat.K"]
    for component_name in ("peak1", "bridge", "peak2"):
        link_at_current_ratio(
            model[f"crab.spectrum.{component_name}.SpecFromDat.K"],
            crab_nebula_k,
        )

if "one_e_1740_compow" in model.sources:
    link_at_current_ratio(
        model.one_e_1740_compow.spectrum.main.composite.K_2,
        model.one_e_1740_compow.spectrum.main.composite.K_1,
    )

if "maxi_j1820" in model.sources:
    link_at_current_ratio(
        model.maxi_j1820.spectrum.main.composite.K_2,
        model.maxi_j1820.spectrum.main.composite.K_1,
    )

if "maxi_j1348" in model.sources:
    link_at_current_ratio(
        model.maxi_j1348.spectrum.main.composite.K_2,
        model.maxi_j1348.spectrum.main.composite.K_1,
    )

if "ngc1068" in model.sources:
    link_at_current_ratio(
        model.ngc1068.spectrum.main.composite.K_2,
        model.ngc1068.spectrum.main.composite.K_1,
    )

link_at_current_ratio(
    model.ngc4151.spectrum.main.composite.K_2,
    model.ngc4151.spectrum.main.composite.K_1,
)

for parameter in model.free_parameters.values():
    set_normalization_step(parameter)

assert len(model.free_parameters) == len(model.sources)
print(
    f"Catalog includes {len(requested_source_names)} explicitly selected "
    "nuisance source entries."
)
print("Catalog sources:")
for source_name in model.sources:
    print(f"  {source_name}")


## Load or bin native source events and fit normalizations

Each cached source file contains the target-GTI-selected
`Em × Phi × PsiChi` histogram on the axes from `agn.yaml`. Before reading
and GTI-binning a native FITS file, the notebook looks for the corresponding
HDF5 file under
`DC4_Sources/DC4_Binned/<GTI_TARGET_NAME>_Cut`.

- If the file exists and `OVERWRITE_BINNED_FILES=False`, it is loaded with
  `Histogram.open`; the GTI events are not binned again.
- If it is missing, the native FITS events are GTI-selected, binned, and saved.
- If `OVERWRITE_BINNED_FILES=True`, non-target caches are regenerated.
- The separately maintained target-source file is loaded through
  `GTI_TARGET_BINNED_FILE` and is never overwritten here.
- Changing `GTI_TARGET_NAME` selects a new `<name>_Cut` directory.

The notebook still reads the native FITS events to construct the independent
full-three-month comparison used to calibrate steady sources. Variable and
flaring sources are calibrated from the loaded target-GTI histograms. No
light-curve-weighted response is used.

In [ ]:
AGN_BINNING_CONFIG = Path(
    "/Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/"
    "continuum_fit/AGN/agn.yaml"
)
VARIABLE_SOURCE_NAMES = {
    "four_c_21p35_noflare",
    "four_c_21p35_flare",
    "maxi_j1820",
    "maxi_j1348",
    "three_c_454p3_low",
    "three_c_454p3_high",
    "psr_b1259",
    "one_rxs_j170849",
    "generic_magnetar",
    "ls_5039",
    "psr_j1846",
    "co_nova_continuum",
}

unknown_variable_sources = VARIABLE_SOURCE_NAMES - set(all_catalog_sources)
if unknown_variable_sources:
    raise RuntimeError(
        "Unknown variable-source classification: "
        + ", ".join(sorted(unknown_variable_sources))
    )

DC4_ROOT_CANDIDATES = [
    Path(
        "/Users/parshadkp/Library/CloudStorage/"
        "OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files"
    ),
    Path(
        "/Users/parshadkp/Library/CloudStorage/"
        "OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/DC4_Files"
    ),
]
DC4_FILES_ROOT = next(
    (candidate for candidate in DC4_ROOT_CANDIDATES if candidate.exists()),
    None,
)
if DC4_FILES_ROOT is None:
    raise FileNotFoundError(
        "Could not find DC4_Files in any configured location: "
        f"{DC4_ROOT_CANDIDATES}"
    )
print("Using DC4 files from:", DC4_FILES_ROOT)
SOURCE_EVENT_ROOT = DC4_FILES_ROOT / "DC4_Sources"

# Change only this source key when calibrating around another target.
# The cache tag and <SOURCE>_Cut folder are derived automatically.
GTI_TARGET_SOURCE_NAME = "ngc4151"
GTI_TARGET_NAME = "".join(
    character
    for character in GTI_TARGET_SOURCE_NAME
    if character.isalnum()
).upper()
GTI_MAX_OFFAXIS = 60.0 * u.deg
GTI_EARTH_OCCULTATION = True

SAVE_BINNED_GTI_FILES = True
LOAD_EXISTING_BINNED_FILES = True
OVERWRITE_BINNED_FILES = False
BINNED_GTI_ROOT = SOURCE_EVENT_ROOT / "DC4_Binned"
BINNED_GTI_DIRECTORY = BINNED_GTI_ROOT / f"{GTI_TARGET_NAME}_Cut"

# Files in this mapping are maintained by their target-data workflows.
# This generator may load them, but never writes or overwrites them.
READ_ONLY_TARGET_BINNED_FILENAMES = {
    "ngc4151": (
        "NGC_4151_ec200_0p30_DC4_COSI_cpl_pl_time_cut_in_fov.hdf5"
    ),
}
read_only_target_filename = READ_ONLY_TARGET_BINNED_FILENAMES.get(
    GTI_TARGET_SOURCE_NAME
)
GTI_TARGET_BINNED_FILE = (
    BINNED_GTI_DIRECTORY / read_only_target_filename
    if read_only_target_filename is not None
    else None
)

ORIENTATION_PATH = (
    DC4_FILES_ROOT
    / "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori"
)
RESPONSE_PATH = (
    DC4_FILES_ROOT
    / "ResponseContinuum.o3.e100_10000.b10log.s10396905069491."
    "m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"
)

SOURCE_EVENT_FILES = {
    "cyg_x1_hard": SOURCE_EVENT_ROOT / "DC3/cygX1_hard_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "crab": SOURCE_EVENT_ROOT / "DC4/Crab_DC4_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "one_e_1740_compow": SOURCE_EVENT_ROOT / "DC3/1E1740_compow_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "grs_1758_258": SOURCE_EVENT_ROOT / "DC3/GRS1758_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "cena": SOURCE_EVENT_ROOT / "DC4/CenA_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "four_c_71p07": SOURCE_EVENT_ROOT / "DC4/4C71p07_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "four_c_21p35_noflare": SOURCE_EVENT_ROOT / "DC3/4C21p35_noflare_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "four_c_21p35_flare": SOURCE_EVENT_ROOT / "DC3/4C21p35_flare_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "maxi_j1820": SOURCE_EVENT_ROOT / "DC3/MAXIJ1820_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "maxi_j1348": SOURCE_EVENT_ROOT / "DC3/MAXIJ1348_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "three_c_454p3_low": SOURCE_EVENT_ROOT / "DC4/3C454p3_low_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "three_c_454p3_high": SOURCE_EVENT_ROOT / "DC4/3C454p3_high_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "ngc1068": SOURCE_EVENT_ROOT / "DC4/NGC_1068_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "ngc4151": SOURCE_EVENT_ROOT / "DC4/NGC_4151_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "cyg_x3": SOURCE_EVENT_ROOT / "DC4/cygX3_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "psr_b1259": SOURCE_EVENT_ROOT / "DC3/PSRB1259_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "one_rxs_j170849": SOURCE_EVENT_ROOT / "DC4/1RXSJ170849_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "generic_magnetar": SOURCE_EVENT_ROOT / "DC4/magnetar2_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "ls_5039": SOURCE_EVENT_ROOT / "DC3/LS5039_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "psr_j1846": SOURCE_EVENT_ROOT / "DC3/PSRJ1846_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
    "co_nova_continuum": SOURCE_EVENT_ROOT / "DC3/nova_co_continuum_3months_unbinned_data_filtered_with_SAAcut.fits.gz",
}

for required_path in (
    ORIENTATION_PATH,
    RESPONSE_PATH,
    *(SOURCE_EVENT_FILES[name] for name in selected_source_names),
):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

# Derive the measured-data axes from the same configuration used by the
# spectral analysis. The time axis is projected out by the likelihood, so the
# native events are accumulated directly in Em x Phi x PsiChi.
binning_configuration = BinnedData(AGN_BINNING_CONFIG)
ENERGY_BIN_EDGES = np.asarray(
    binning_configuration.energy_bins,
    dtype=float,
)
PHI_BIN_EDGES = np.linspace(
    0.0,
    180.0,
    int(180.0 / binning_configuration.phi_pix_size) + 1,
)

measurement_axes = Axes(
    [
        Axis(ENERGY_BIN_EDGES, unit=u.keV, label="Em"),
        Axis(PHI_BIN_EDGES, unit=u.deg, label="Phi"),
        HealpixAxis(
            nside=binning_configuration.nside,
            scheme=binning_configuration.scheme,
            coordsys="galactic",
            label="PsiChi",
        ),
    ],
    copy_axes=False,
)
comparison_data = EmCDSBinnedData(
    Histogram(measurement_axes, sparse=True)
)

print("FITS/model comparison axes from:", AGN_BINNING_CONFIG)
print("  Em bins:", measurement_axes["Em"].nbins)
print("  Phi bins:", measurement_axes["Phi"].nbins)
print("  PsiChi pixels:", measurement_axes["PsiChi"].nbins)

if GTI_TARGET_SOURCE_NAME not in all_catalog_sources:
    raise ValueError(
        f"Unknown GTI target source: {GTI_TARGET_SOURCE_NAME!r}"
    )
gti_target_source = all_catalog_sources[GTI_TARGET_SOURCE_NAME]
gti_target_coord = SkyCoord(
    l=gti_target_source.position.l.value * u.deg,
    b=gti_target_source.position.b.value * u.deg,
    frame="galactic",
)

spacecraft_history_full = SpacecraftHistory.open(ORIENTATION_PATH)
comparison_gti = GoodTimeInterval.from_pointing_cut(
    gti_target_coord,
    spacecraft_history_full,
    GTI_MAX_OFFAXIS,
    earth_occ=GTI_EARTH_OCCULTATION,
)
spacecraft_history = spacecraft_history_full.apply_gti(comparison_gti)

detector_response = FullDetectorResponse.open(str(RESPONSE_PATH))
instrument_response = BinnedInstrumentResponse(
    detector_response,
    comparison_data,
)


def make_point_source_response(history):
    return BinnedThreeMLPointSourceResponse(
        data=comparison_data,
        instrument_response=instrument_response,
        sc_history=history,
        energy_axis=detector_response.axes["Ei"],
        polarization_axis=(
            detector_response.axes["Pol"]
            if "Pol" in detector_response.axes.labels
            else None
        ),
        nside=2 * comparison_data.axes["PsiChi"].nside,
    )


gti_folded_response = BinnedThreeMLModelFolding(
    data=comparison_data,
    point_source_response=make_point_source_response(spacecraft_history),
)
full_folded_response = BinnedThreeMLModelFolding(
    data=comparison_data,
    point_source_response=make_point_source_response(
        spacecraft_history_full
    ),
)
print(
    f"{GTI_TARGET_NAME} GTI livetime:",
    spacecraft_history.cumulative_livetime(),
)
print(
    "Full three-month livetime:",
    spacecraft_history_full.cumulative_livetime(),
)
selected_variable_sources = [
    name for name in selected_source_names if name in VARIABLE_SOURCE_NAMES
]
selected_steady_sources = [
    name for name in selected_source_names if name not in VARIABLE_SOURCE_NAMES
]
print("Full-duration normalization:", selected_steady_sources)
print(f"{GTI_TARGET_NAME} GTI-averaged normalization:", selected_variable_sources)
print("No light-curve-weighted response is used.")

print("GTI binned-file directory:", BINNED_GTI_DIRECTORY)
print("Load existing binned files:", LOAD_EXISTING_BINNED_FILES)
print("Overwrite existing non-target files:", OVERWRITE_BINNED_FILES)
print(
    "Target binned file:",
    (
        f"{GTI_TARGET_BINNED_FILE} (read only)"
        if GTI_TARGET_BINNED_FILE is not None
        else "generated like the other source caches"
    ),
)


In [ ]:
def events_in_gti(times, gti):
    """Vectorized GTI membership that is safe for unsorted event times."""
    times = np.asarray(times, dtype=float)
    starts = np.asarray(gti.tstart_list.unix, dtype=float)
    stops = np.asarray(gti.tstop_list.unix, dtype=float)

    interval_index = np.searchsorted(starts, times, side="right") - 1
    valid = interval_index >= 0
    selected = np.zeros(times.size, dtype=bool)
    selected[valid] = times[valid] < stops[interval_index[valid]]
    return selected


def binned_gti_output_path(source_name, event_path):
    """Return the configured cache path for one target-GTI histogram."""
    if (
        source_name == GTI_TARGET_SOURCE_NAME
        and GTI_TARGET_BINNED_FILE is not None
    ):
        return GTI_TARGET_BINNED_FILE

    filename = Path(event_path).name
    for suffix in (".fits.gz", ".fits", ".gz"):
        if filename.endswith(suffix):
            filename = filename[: -len(suffix)]
            break
    filename = filename.replace("unbinned_data", "binned_data")
    return (
        BINNED_GTI_DIRECTORY
        / f"{filename}_{GTI_TARGET_NAME}_Cut.hdf5"
    )


def validate_cached_gti_histogram(histogram, cache_path, expected_axes):
    """Fail clearly if a cache does not match the agn.yaml analysis binning."""
    expected_labels = tuple(expected_axes.labels)
    actual_labels = tuple(histogram.axes.labels)
    if actual_labels != expected_labels:
        raise ValueError(
            f"{cache_path} has axes {actual_labels}, expected "
            f"{expected_labels} from {AGN_BINNING_CONFIG}"
        )

    expected_bins = tuple(expected_axes[label].nbins for label in expected_labels)
    actual_bins = tuple(histogram.axes[label].nbins for label in actual_labels)
    if actual_bins != expected_bins:
        raise ValueError(
            f"{cache_path} has bin counts {actual_bins}, expected "
            f"{expected_bins} from {AGN_BINNING_CONFIG}"
        )
    return histogram


def load_cached_gti_histogram(source_name, event_path, axes):
    """Load an existing target-GTI histogram when cache use is enabled."""
    cache_path = binned_gti_output_path(source_name, event_path)
    if not LOAD_EXISTING_BINNED_FILES or not cache_path.exists():
        return None, cache_path

    # The target file is read-only here. For other sources, an explicit
    # overwrite request forces regeneration from the native events.
    is_read_only_target = (
        source_name == GTI_TARGET_SOURCE_NAME
        and GTI_TARGET_BINNED_FILE is not None
    )
    if not is_read_only_target and OVERWRITE_BINNED_FILES:
        return None, cache_path

    histogram = Histogram.open(cache_path)
    return (
        validate_cached_gti_histogram(histogram, cache_path, axes),
        cache_path,
    )


def save_gti_histogram(source_name, cache_path, histogram):
    """Save a new non-target GTI cache without replacing files by default."""
    if (
        source_name == GTI_TARGET_SOURCE_NAME
        and GTI_TARGET_BINNED_FILE is not None
    ):
        return "read-only target file missing; binned in memory only"
    if not SAVE_BINNED_GTI_FILES:
        return "saving disabled"

    output_existed = cache_path.exists()
    if output_existed and not OVERWRITE_BINNED_FILES:
        return "kept existing file"

    BINNED_GTI_DIRECTORY.mkdir(parents=True, exist_ok=True)
    histogram.write(cache_path, overwrite=OVERWRITE_BINNED_FILES)
    return "overwritten" if output_existed else "written"


def load_or_bin_native_source_fits(source_name, event_path, axes, gti):
    """Load the GTI cache when available; always retain the full comparison."""
    gti_histogram, cache_path = load_cached_gti_histogram(
        source_name,
        event_path,
        axes,
    )
    loaded_gti_cache = gti_histogram is not None

    with fits.open(event_path, memmap=False) as hdul:
        events = hdul[1].data
        times = np.asarray(events["TimeTags"], dtype=float)
        energies = np.asarray(events["Energies"], dtype=float)
        phi = np.rad2deg(np.asarray(events["Phi"], dtype=float))
        psi_chi = SkyCoord(
            l=np.asarray(events["Chi galactic"], dtype=float) * u.deg,
            b=np.asarray(events["Psi galactic"], dtype=float) * u.deg,
            frame="galactic",
        )

        full_histogram = Histogram(axes, sparse=True)
        full_histogram.fill(
            energies * u.keV,
            phi * u.deg,
            psi_chi,
        )

        if not loaded_gti_cache:
            selected = events_in_gti(times, gti)
            gti_histogram = Histogram(axes, sparse=True)
            if np.any(selected):
                gti_histogram.fill(
                    energies[selected] * u.keV,
                    phi[selected] * u.deg,
                    psi_chi[selected],
                )

    if loaded_gti_cache:
        cache_status = (
            "loaded existing read-only target file"
            if (
                source_name == GTI_TARGET_SOURCE_NAME
                and GTI_TARGET_BINNED_FILE is not None
            )
            else "loaded existing file"
        )
    else:
        cache_status = save_gti_histogram(
            source_name,
            cache_path,
            gti_histogram,
        )

    return full_histogram, gti_histogram, cache_path, cache_status


def histogram_counts(histogram):
    return float(
        histogram.project("Em").to_dense(copy=False).contents.sum()
    )


def source_normalization_parameter(source_model, source_name):
    """Return the one independent normalization amplitude for a source."""
    candidates = [
        (path, parameter)
        for path, parameter in source_model.free_parameters.items()
        if path.startswith(f"{source_name}.")
        and parameter.is_normalization
    ]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one free normalization for {source_name!r}; "
            f"found {[path for path, _ in candidates]}"
        )
    return candidates[0]


def poisson_deviance(observed_histogram, expected_histogram):
    """Poisson deviance; normalization cannot remove shape discrepancies."""
    observed = np.asarray(
        observed_histogram.to_dense(copy=False).contents,
        dtype=float,
    )
    expected = np.asarray(
        expected_histogram.to_dense(copy=False).contents,
        dtype=float,
    )

    unsupported = (observed > 0) & (expected <= 0)
    if np.any(unsupported):
        return np.inf, int(np.count_nonzero(unsupported))

    positive = observed > 0
    terms = expected - observed
    terms[positive] += observed[positive] * np.log(
        observed[positive] / expected[positive]
    )
    return 2.0 * float(np.sum(terms)), 0


def folded_source_histogram(source_model, folded_response):
    folded_response.set_model(source_model)
    return folded_response.expectation()


full_count_comparison = {}
gti_count_comparison = {}
normalization_fit_rows = []
binned_cache_rows = []

for source_name in selected_source_names:
    fit_scope = (
        f"{GTI_TARGET_NAME} GTI"
        if source_name in VARIABLE_SOURCE_NAMES
        else "full three months"
    )
    source_copy = deepcopy(model.sources[source_name])

    # The current continuum response has no polarization axis. Keep the saved
    # Crab catalog polarized, but use an unpolarized copy for calibration.
    if source_name == "crab" and "Pol" not in detector_response.axes.labels:
        for component in source_copy.components.values():
            component.polarization.degree.value = 0.0

    source_model = Model(source_copy)
    source_norm_path, source_norm = source_normalization_parameter(
        source_model,
        source_name,
    )
    catalog_norm_path, catalog_norm = source_normalization_parameter(
        model,
        source_name,
    )

    (
        fits_full_hist,
        fits_gti_hist,
        binned_cache_path,
        binned_cache_status,
    ) = load_or_bin_native_source_fits(
        source_name,
        SOURCE_EVENT_FILES[source_name],
        measurement_axes,
        comparison_gti,
    )

    binned_cache_rows.append(
        {
            "source": source_name,
            "GTI counts": histogram_counts(fits_gti_hist),
            "status": binned_cache_status,
            "file": (
                str(binned_cache_path)
                if binned_cache_path is not None
                else "—"
            ),
        }
    )

    modeled_gti_before = folded_source_histogram(
        source_model,
        gti_folded_response,
    )
    modeled_full_before = folded_source_histogram(
        source_model,
        full_folded_response,
    )

    if source_name in VARIABLE_SOURCE_NAMES:
        fit_data_hist = fits_gti_hist
        fit_model_hist = modeled_gti_before
    else:
        fit_data_hist = fits_full_hist
        fit_model_hist = modeled_full_before

    fits_fit_counts = histogram_counts(fit_data_hist)
    model_fit_counts = histogram_counts(fit_model_hist)
    normalization_before = float(catalog_norm.value)

    if model_fit_counts > 0 and fits_fit_counts > 0:
        # Analytic Poisson MLE for one multiplicative source amplitude.
        fitted_scale = fits_fit_counts / model_fit_counts
        source_norm.value *= fitted_scale
        catalog_norm.value *= fitted_scale
        fit_status = "fitted"
    elif model_fit_counts > 0 and fits_fit_counts == 0:
        # A zero-count transient has an MLE at zero. Astromodels positive
        # normalizations use logarithmic transformations, so represent zero
        # by the smallest allowed positive value.
        positive_floors = [
            value
            for value in (
                source_norm.min_value,
                catalog_norm.min_value,
                1e-30,
            )
            if value is not None and value > 0
        ]
        frozen_floor = max(positive_floors)
        fitted_scale = frozen_floor / source_norm.value
        source_norm.value = frozen_floor
        catalog_norm.value = frozen_floor
        fit_status = "0 FITS counts; frozen at normalization floor"
    elif fits_fit_counts == 0:
        fitted_scale = 1.0
        fit_status = "unconstrained (0 FITS and 0 model counts)"
    else:
        raise RuntimeError(
            f"{source_name}: native FITS has {fits_fit_counts} counts but "
            "the forward-folded model predicts zero; a normalization-only "
            "fit is impossible."
        )

    set_normalization_step(catalog_norm)

    modeled_gti_after = folded_source_histogram(
        source_model,
        gti_folded_response,
    )
    modeled_full_after = folded_source_histogram(
        source_model,
        full_folded_response,
    )

    fit_after_hist = (
        modeled_gti_after
        if source_name in VARIABLE_SOURCE_NAMES
        else modeled_full_after
    )
    deviance, unsupported_bins = poisson_deviance(
        fit_data_hist,
        fit_after_hist,
    )

    normalization_fit_rows.append(
        {
            "source": source_name,
            "fit scope": fit_scope,
            "normalization parameter": catalog_norm_path,
            "normalization before": normalization_before,
            "fitted scale": fitted_scale,
            "frozen normalization": float(catalog_norm.value),
            "FITS fit counts": fits_fit_counts,
            "model before": model_fit_counts,
            "model after": histogram_counts(fit_after_hist),
            "after/FITS": (
                histogram_counts(fit_after_hist) / fits_fit_counts
                if fits_fit_counts > 0
                else np.nan
            ),
            "Poisson deviance": deviance,
            "unsupported FITS bins": unsupported_bins,
            "status": fit_status,
        }
    )

    full_count_comparison[source_name] = {
        "before": np.asarray(
            modeled_full_before.project("Em")
            .to_dense(copy=False).contents,
            dtype=float,
        ),
        "after": np.asarray(
            modeled_full_after.project("Em")
            .to_dense(copy=False).contents,
            dtype=float,
        ),
        "fits": np.asarray(
            fits_full_hist.project("Em").to_dense(copy=False).contents,
            dtype=float,
        ),
    }
    gti_count_comparison[source_name] = {
        "before": np.asarray(
            modeled_gti_before.project("Em")
            .to_dense(copy=False).contents,
            dtype=float,
        ),
        "after": np.asarray(
            modeled_gti_after.project("Em")
            .to_dense(copy=False).contents,
            dtype=float,
        ),
        "fits": np.asarray(
            fits_gti_hist.project("Em").to_dense(copy=False).contents,
            dtype=float,
        ),
    }

normalization_fit_table = (
    pd.DataFrame(normalization_fit_rows).set_index("source")
)
binned_cache_table = (
    pd.DataFrame(binned_cache_rows).set_index("source")
)

# Keep the detailed plots focused on the NGC 4151 GTI comparison.
count_comparison = gti_count_comparison


## Freeze and save the hybrid catalog

The table records each source's fit scope, original amplitude, fitted scale,
and YAML amplitude. Every parameter is frozen before saving. The filename
includes `NGC4151_GTI` because the variable-source amplitudes are specific to
this time selection, although the steady-source amplitudes use the full
three-month observation.


In [ ]:
display(
    binned_cache_table.style.format(
        {"GTI counts": "{:,.0f}"}
    )
)
print(
    "Binned GTI files written or reused:",
    int((binned_cache_table["status"] != "saving disabled").sum())
    - int(
        binned_cache_table["status"]
        .str.startswith("skipped GTI target")
        .sum()
    ),
)

display(
    normalization_fit_table.style.format(
        {
            "normalization before": "{:.6g}",
            "fitted scale": "{:.6g}",
            "frozen normalization": "{:.6g}",
            "FITS fit counts": "{:,.0f}",
            "model before": "{:,.1f}",
            "model after": "{:,.1f}",
            "after/FITS": "{:.6f}",
            "Poisson deviance": "{:,.1f}",
            "unsupported FITS bins": "{:,.0f}",
        },
        na_rep="—",
    )
)

catalog_directory = Path(
    "/Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/"
    "continuum_fit/AGN"
)
source_count = len(model.sources)
source_count_label = (
    f"{source_count}source"
    if source_count == 1
    else f"{source_count}sources"
)
catalog_path = (
    catalog_directory
    / (
        f"source_catalog_DC4_{source_count_label}_"
        f"{GTI_TARGET_NAME}_GTI_fit_norm.yaml"
    )
)

for parameter in list(model.free_parameters.values()):
    parameter.fix = True

assert len(model.free_parameters) == 0
model.save(catalog_path, overwrite=True)

print(
    f"Saved hybrid FITS-normalized fixed {len(model.sources)}-entry catalog: "
    f"{catalog_path}"
)


In [ ]:
# Required in any fresh process before load_model:
from cosipy.threeml.custom_functions import SpecFromDat  # noqa: F401, E402

reloaded_model = load_model(catalog_path)
assert list(reloaded_model.sources) == selected_source_names
assert len(reloaded_model.free_parameters) == 0

for source_name, row in normalization_fit_table.iterrows():
    saved_parameter = reloaded_model[row["normalization parameter"]]
    np.testing.assert_allclose(
        saved_parameter.value,
        row["frozen normalization"],
        rtol=1e-10,
    )
    assert saved_parameter.fix

print("Reloaded sources:", list(reloaded_model.sources))
print("All fitted catalog normalizations and shape parameters are fixed.")


In [ ]:
energy = np.geomspace(ENERGY_MIN, ENERGY_MAX, 500)

fig, ax = plt.subplots(figsize=(10, 7))
for source_name, source in model.sources.items():
    ax.loglog(energy, source(energy), label=source_name)

ax.set_xlabel("Energy (keV)")
ax.set_ylabel(r"Differential photon flux (ph cm$^{-2}$ s$^{-1}$ keV$^{-1}$)")
ax.set_xlim(ENERGY_MIN, ENERGY_MAX)
ax.legend(
    fontsize=7,
    ncol=3,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.16),
)
ax.grid(alpha=0.25, which="both")
fig.tight_layout()
plt.show()

### NGC 4151 GTI-selected counts

Variable and flaring sources are fitted in this selection, so their post-fit
totals should match their GTI-filtered native FITS totals. Steady sources are
fitted over the full three months; their GTI rows are an independent validation
of the response and sky selection. Differences among individual energy or
Compton-data-space bins are shape differences that one amplitude cannot remove.


In [ ]:
gti_table_rows = []
for source_name in selected_source_names:
    counts = gti_count_comparison[source_name]
    fits_total = counts["fits"].sum()
    before_total = counts["before"].sum()
    after_total = counts["after"].sum()
    gti_table_rows.append(
        {
            "source": source_name,
            "FITS GTI counts": int(fits_total),
            "model before": before_total,
            "before/FITS": (
                before_total / fits_total if fits_total > 0 else np.nan
            ),
            "model after": after_total,
            "after/FITS": (
                after_total / fits_total if fits_total > 0 else np.nan
            ),
        }
    )

gti_counts_table = pd.DataFrame(gti_table_rows).set_index("source")
display(
    gti_counts_table.style.format(
        {
            "FITS GTI counts": "{:,.0f}",
            "model before": "{:,.1f}",
            "before/FITS": "{:.3f}",
            "model after": "{:,.1f}",
            "after/FITS": "{:.6f}",
        },
        na_rep="—",
    )
)


### Full three-month source-file counts

Steady sources are fitted in this selection, so their post-fit totals should
match the full native FITS totals. Variable and flaring sources are fitted only
inside the NGC 4151 GTI; their full-duration rows are out-of-fit diagnostics and
are not expected to match.


In [ ]:
full_table_rows = []
for source_name in selected_source_names:
    counts = full_count_comparison[source_name]
    fits_total = counts["fits"].sum()
    before_total = counts["before"].sum()
    after_total = counts["after"].sum()
    full_table_rows.append(
        {
            "source": source_name,
            "FITS full counts": int(fits_total),
            "model before": before_total,
            "before/FITS": (
                before_total / fits_total if fits_total > 0 else np.nan
            ),
            "model after": after_total,
            "after/FITS": (
                after_total / fits_total if fits_total > 0 else np.nan
            ),
        }
    )

full_counts_table = pd.DataFrame(full_table_rows).set_index("source")
display(
    full_counts_table.style.format(
        {
            "FITS full counts": "{:,.0f}",
            "model before": "{:,.1f}",
            "before/FITS": "{:.3f}",
            "model after": "{:,.1f}",
            "after/FITS": "{:.3f}",
        },
        na_rep="—",
    )
)


In [ ]:
source_names = list(count_comparison)
fits_totals = np.array(
    [count_comparison[name]["fits"].sum() for name in source_names]
)
before_totals = np.array(
    [count_comparison[name]["before"].sum() for name in source_names]
)
after_totals = np.array(
    [count_comparison[name]["after"].sum() for name in source_names]
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

positive_before = (fits_totals > 0) & (before_totals > 0)
positive_after = (fits_totals > 0) & (after_totals > 0)
axes[0].scatter(
    fits_totals[positive_before],
    before_totals[positive_before],
    label="Before FITS normalization",
    alpha=0.75,
)
axes[0].scatter(
    fits_totals[positive_after],
    after_totals[positive_after],
    label="After FITS normalization",
    marker="x",
)
positive_values = np.concatenate(
    (
        fits_totals[fits_totals > 0],
        before_totals[before_totals > 0],
        after_totals[after_totals > 0],
    )
)
comparison_min = positive_values.min()
comparison_max = positive_values.max()
axes[0].plot(
    [comparison_min, comparison_max],
    [comparison_min, comparison_max],
    color="black",
    linestyle=":",
    label="1:1",
)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("Binned native FITS counts")
axes[0].set_ylabel("Forward-folded model counts")
axes[0].legend()
axes[0].grid(alpha=0.25, which="both")

before_ratios = np.divide(
    before_totals,
    fits_totals,
    out=np.full_like(before_totals, np.nan, dtype=float),
    where=fits_totals > 0,
)
after_ratios = np.divide(
    after_totals,
    fits_totals,
    out=np.full_like(after_totals, np.nan, dtype=float),
    where=fits_totals > 0,
)
x_positions = np.arange(len(source_names))
bar_width = 0.4
axes[1].bar(
    x_positions - bar_width / 2,
    before_ratios,
    width=bar_width,
    label="Before",
)
axes[1].bar(
    x_positions + bar_width / 2,
    after_ratios,
    width=bar_width,
    label="After",
)
axes[1].axhline(1.0, color="black", linestyle=":")
axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(source_names, rotation=90)
axes[1].set_ylabel("Forward-folded model / FITS counts")
axes[1].legend()
axes[1].grid(alpha=0.25, axis="y")
plt.show()


# Change this name to inspect any selected source's reconstructed-energy shape.
COUNT_SPECTRUM_SOURCE_NAME = "ngc4151"
if COUNT_SPECTRUM_SOURCE_NAME not in count_comparison:
    raise ValueError(
        f"{COUNT_SPECTRUM_SOURCE_NAME!r} is not in selected_source_names"
    )

detailed_counts = count_comparison[COUNT_SPECTRUM_SOURCE_NAME]
energy_centers = np.sqrt(
    ENERGY_BIN_EDGES[:-1] * ENERGY_BIN_EDGES[1:]
)

fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
ax.stairs(
    detailed_counts["before"],
    ENERGY_BIN_EDGES,
    color="royalblue",
    linestyle="--",
    label="Model before",
)
ax.stairs(
    detailed_counts["after"],
    ENERGY_BIN_EDGES,
    color="red",
    label="Model after",
)
ax.stairs(
    detailed_counts["fits"],
    ENERGY_BIN_EDGES,
    color="black",
    linestyle=":",
    label="Binned native FITS",
)
ax.errorbar(
    energy_centers,
    detailed_counts["fits"],
    yerr=np.sqrt(detailed_counts["fits"]),
    color="black",
    linewidth=0,
    elinewidth=1,
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Measured energy (keV)")
ax.set_ylabel("Counts")
ax.set_title(COUNT_SPECTRUM_SOURCE_NAME)
ax.legend()
ax.grid(alpha=0.25, which="both")
plt.show()


## Run the selected-source fit

Open `NGC4151_DC4_Spectral_Fit_Source_Cat_fit_norm.ipynb` after this notebook
finishes. It loads the generated hybrid catalog and uses the same ordinary
NGC 4151 GTI response. It does not construct light-curve-weighted responses,
because the variable-source amplitudes already include their GTI duty cycles.

To test a different catalog, return here, edit `ADDITIONAL_SOURCE_NAMES`, and
rerun all cells. Do not manually free nuisance-source parameters in the
generator; the YAML remains a fixed baseline catalog.


### Selection workflow

1. Set `ADDITIONAL_SOURCE_NAMES` to the nuisance sources to include.
2. Run the notebook from the top. Existing target-GTI HDF5 files are loaded
   first; only missing caches are GTI-binned from native FITS events using the
   `agn.yaml` axes.
3. Steady sources use full three-month counts and the full response.
4. Variable/flaring sources use NGC 4151 GTI counts and the ordinary GTI
   response; their amplitudes absorb duration and duty cycle.
5. All parameters are frozen and the notebook writes
   `source_catalog_DC4_<N>sources_NGC4151_GTI_fit_norm.yaml`.
6. The spectral-fitting notebook must use the same ordinary GTI response and
   must not apply the light curves again.

Change only `GTI_TARGET_SOURCE_NAME` to use another target. Its sanitized,
uppercase source key selects a new `<SOURCE>_Cut` folder automatically.
Set `OVERWRITE_BINNED_FILES=True` only when you intentionally want to
regenerate writable cache files. Target files listed in
`READ_ONLY_TARGET_BINNED_FILENAMES` remain read-only.

The variable-source part of this catalog is selection-specific. Refit those
amplitudes for a different GTI. Matching total counts does not guarantee
agreement in every `Em`, `Phi`, or `PsiChi` bin; use the deviance and
detailed plots to assess remaining shape differences.
